# Task 3 — Multimodal fusion modelling
**Md Soad Solaiman · a1978974**

We compare individual sensor inputs with two ways of combining information. All models use the unchanged Task 2 features, eligibility rules and held-out participant folds. No test result is used to choose parameters.

Run this notebook from the project folder after Tasks 1–2. Required inputs are already included in `results/`. This notebook does not need the raw ZIP. Task 4 reads its saved predictions, so it does not retrain models.

The original assignment brief/rubric was not supplied. This work follows the requirements discussed in the conversation, with fixed parameters instead of hyperparameter search. Formal rubric compliance remains unverified.

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.evaluation import CLASSES, scores

RESULTS = REPO_ROOT / 'results'
GENERATED_NOTES = RESULTS / 'generated_notes'
GENERATED_NOTES.mkdir(parents=True, exist_ok=True)
SEED = 42
X = pd.read_csv(RESULTS / 'window_features.csv.gz', index_col='window_id')
metadata = pd.read_csv(RESULTS / 'window_metadata.csv').set_index('window_id')
assert X.index.is_unique and metadata.index.is_unique
assert set(X.index) == set(metadata.index)
metadata = metadata.loc[X.index]
dictionary = pd.read_csv(RESULTS / 'feature_dictionary.csv')
folds = pd.read_csv(RESULTS / 'participant_folds.csv')
assert np.isfinite(X.to_numpy()).all()
assert not set(['activity', 'participant', 'segment', 'start_sample', 'end_sample']).intersection(X.columns)
assert set(metadata.activity) == set(CLASSES)
assert set(folds.test_participant) == set(metadata.participant)
print(f'{len(X):,} matched windows, {len(folds)} held-out participant folds')

4,581 matched windows, 10 held-out participant folds


## 3.1 Select sources and modalities
We use the **chest, left ankle and right wrist/lower-arm devices**: three physical sources. We select **acceleration** from all three and **angular velocity (gyroscope)** from ankle and wrist: two modalities, 15 channels and 90 extracted features. The chest device has no gyroscope channels in this dataset.

Chest acceleration describes trunk movement and posture; ankle acceleration/rotation describes lower-limb movement; wrist acceleration/rotation describes arm movement. These are reasons to test complementary inputs, not evidence that any source improves predictions. Acceleration and angular velocity are different physical measurements, even though both describe motion.

ECG and magnetometer features are excluded to keep this comparison focused on movement and avoid adding specialised ECG or magnetic-field interpretation. This is a scope choice made before modelling, not a finding that these modalities are inferior. The Task 2 **all-channel eligibility rule remains unchanged**, including exclusions caused by these unused channels. Results therefore apply only to that common retained population.

The modality baselines pool locations within one modality. The three additional accelerometer baselines show the contribution of each individual source. A multi-location accelerometer model is still unimodal.

In [2]:
def select_features(source=None, modality=None):
    selected = dictionary.copy()
    if source is not None:
        selected = selected[selected.source == source]
    if modality is not None:
        selected = selected[selected.modality == modality]
    return selected.feature.tolist()

feature_sets = {
    'Chest accelerometer': select_features('chest', 'acc'),
    'Ankle accelerometer': select_features('ankle', 'acc'),
    'Wrist accelerometer': select_features('wrist', 'acc'),
    'Accelerometers (all locations)': select_features(modality='acc'),
    'Gyroscopes (ankle + wrist)': select_features(modality='gyro')}
ACC = 'Accelerometers (all locations)'
GYRO = 'Gyroscopes (ankle + wrist)'
FEATURE = 'Feature fusion'
DECISION = 'Decision fusion'
feature_sets[FEATURE] = feature_sets[ACC] + feature_sets[GYRO]
assert len(feature_sets[FEATURE]) == 90
assert len(set(feature_sets[FEATURE])) == 90
assert set(feature_sets[ACC]).isdisjoint(feature_sets[GYRO])
assert set(dictionary.set_index('feature').loc[feature_sets[FEATURE], 'source']) == {'chest', 'ankle', 'wrist'}
assert set(dictionary.set_index('feature').loc[feature_sets[FEATURE], 'modality']) == {'acc', 'gyro'}
model_rows = []
for name, columns in feature_sets.items():
    info = dictionary.set_index('feature').loc[columns]
    model_rows.append({'model': name, 'features': len(columns),
                       'sources': info.source.nunique(), 'modalities': info.modality.nunique(),
                       'component_models': 1})
model_rows.append({'model': DECISION, 'features': 90, 'sources': 3,
                   'modalities': 2, 'component_models': 2})
manifest = pd.DataFrame(model_rows)
manifest.to_csv(RESULTS / 'task3_experiment_manifest.csv', index=False)
(RESULTS / 'task3_feature_sets.json').write_text(json.dumps(feature_sets, indent=2))
display(manifest)

,model,features,sources,modalities,component_models
0,Chest accelerometer,18,1,1,1
1,Ankle accelerometer,18,1,1,1
2,Wrist accelerometer,18,1,1,1
3,Accelerometers (all locations),54,3,1,1
4,Gyroscopes (ankle + wrist),36,2,1,1
5,Feature fusion,90,3,2,1
6,Decision fusion,90,3,2,2


## 3.2 Fix the model settings before testing
Every fitted model is a Random Forest with **200 trees**, `max_features='sqrt'`, `min_samples_leaf=2`, unlimited tree depth and training-derived balanced class weights. We use seed 42 and one worker for a consistent timing procedure. Two samples per leaf mildly limit very small leaves; 200 trees provide an ensemble without an extensive search. These are practical fixed choices, not claimed optimal settings.

There is **no hyperparameter search**, so inner validation is not performed or claimed. If parameters are tuned in future, tuning must be grouped by participant within each outer training fold. Scaling is unnecessary for these tree splits and Task 2 produced finite features, so no new imputer or scaler is fitted. Class weights are calculated from training labels only.

**Feature fusion** concatenates the 54 acceleration features and 36 gyroscope features before fitting one forest. A tree may split on features from both modalities.

**Decision fusion** independently fits the acceleration and gyroscope forests, aligns their probability columns to activity labels 1–12, and uses:

`fused_probability = 0.5 * acceleration_probability + 0.5 * gyroscope_probability`

The class with the largest fused probability is predicted. Equal weights are fixed; test labels never enter this calculation. Ties use the first class in the fixed ascending order. Probability calibration and sensor-dropout robustness are not tested.

In [3]:
PARAMETERS = dict(n_estimators=200, max_depth=None, min_samples_leaf=2,
                  max_features='sqrt', class_weight='balanced',
                  random_state=SEED, n_jobs=1)
config = {'parameters': PARAMETERS, 'tuning': 'none; fixed before outer evaluation',
          'outer_evaluation': 'leave one participant out', 'classes': CLASSES.tolist(),
          'decision_components': [ACC, GYRO], 'decision_weights': [0.5, 0.5],
          'python': platform.python_version(), 'sklearn': sklearn.__version__,
          'numpy': np.__version__, 'pandas': pd.__version__,
          'machine_architecture': platform.machine(),
          'feature_sha256': hashlib.sha256((RESULTS / 'window_features.csv.gz').read_bytes()).hexdigest(),
          'metadata_sha256': hashlib.sha256((RESULTS / 'window_metadata.csv').read_bytes()).hexdigest()}
# Save the choices before training starts.
(RESULTS / 'task3_model_config.json').write_text(json.dumps(config, indent=2))

def aligned_probabilities(model, values):
    raw = model.predict_proba(values)
    probability = np.zeros((len(values), len(CLASSES)))
    for column, label in enumerate(model.classes_):
        probability[:, np.flatnonzero(CLASSES == label)[0]] = raw[:, column]
    assert np.allclose(probability.sum(axis=1), 1)
    return probability

## 3.3 Run the held-out participant experiments
Each fold trains on nine participants and predicts only the tenth. Feature extraction and offline gap filling were performed within individual participant/activity segments in Task 2; they did not learn population statistics. Activity-boundary knowledge remains an explicit deployment limitation.

Training time measures `fit`. Prediction time includes class-probability calculation and alignment. For decision fusion, we sum the two component training times and prediction times and add the measured averaging time. The components are reused from their baseline runs, so these are **accounted standalone deployment costs**, not additional duplicate fitting. Timing excludes file loading, feature extraction, metrics and output writing. It is a single measurement per fold on this execution environment; no memory, model-size or battery measurement is claimed.

In [4]:
prediction_frames, runtime_rows, fold_rows, split_rows = [], [], [], []
y = metadata.activity.to_numpy()
for fold in folds.itertuples(index=False):
    test = (metadata.participant == fold.test_participant).to_numpy()
    train = ~test
    assert int(test.sum()) == fold.test_windows
    assert int(train.sum()) == fold.train_windows
    training_people = sorted(metadata.loc[train, 'participant'].unique().tolist())
    assert set(training_people).isdisjoint({fold.test_participant})
    assert training_people == list(map(int, str(fold.train_participants).split()))
    assert set(y[train]) == set(CLASSES)
    split_rows.append({'fold': int(fold.fold), 'test_participant': int(fold.test_participant),
                       'training_participants': training_people,
                       'train_windows': int(train.sum()), 'test_windows': int(test.sum())})
    probabilities, costs = {}, {}
    for name, columns in feature_sets.items():
        values = X[columns].to_numpy(dtype=np.float32)
        train_values, test_values = values[train], values[test]
        model = RandomForestClassifier(**PARAMETERS)
        start = perf_counter()
        model.fit(train_values, y[train])
        train_seconds = perf_counter() - start
        start = perf_counter()
        probabilities[name] = aligned_probabilities(model, test_values)
        prediction_seconds = perf_counter() - start
        costs[name] = (train_seconds, prediction_seconds)
    start = perf_counter()
    probabilities[DECISION] = (probabilities[ACC] + probabilities[GYRO]) / 2
    averaging_seconds = perf_counter() - start
    costs[DECISION] = (costs[ACC][0] + costs[GYRO][0],
                       costs[ACC][1] + costs[GYRO][1] + averaging_seconds)
    for name, probability in probabilities.items():
        assert probability.shape == (test.sum(), 12)
        assert np.allclose(probability.sum(axis=1), 1)
        predicted = CLASSES[probability.argmax(axis=1)]
        frame = metadata.loc[test, ['participant', 'activity']].reset_index()
        frame = frame.rename(columns={'activity': 'true_label'})
        frame['model'], frame['fold'], frame['predicted_label'] = name, fold.fold, predicted
        for j, label in enumerate(CLASSES):
            frame[f'p_{label}'] = probability[:, j]
        prediction_frames.append(frame)
        fold_rows.append({'model': name, 'participant': fold.test_participant,
                          'fold': fold.fold, 'windows': int(test.sum()), **scores(y[test], predicted)})
        train_seconds, prediction_seconds = costs[name]
        runtime_rows.append({'model': name, 'fold': fold.fold,
                             'train_seconds': train_seconds, 'prediction_seconds': prediction_seconds,
                             'total_seconds': train_seconds + prediction_seconds})
    print(f'Fold {fold.fold}/10 complete: participant {fold.test_participant}, {test.sum()} test windows')

predictions = pd.concat(prediction_frames, ignore_index=True)
fold_metrics = pd.DataFrame(fold_rows)
runtimes = pd.DataFrame(runtime_rows)
predictions.to_csv(RESULTS / 'task3_oof_predictions.csv.gz', index=False, compression='gzip')
fold_metrics.to_csv(RESULTS / 'task3_fold_metrics.csv', index=False)
runtimes.to_csv(RESULTS / 'task3_fold_runtimes.csv', index=False)
(RESULTS / 'task3_split_audit.json').write_text(json.dumps(split_rows, indent=2))

Fold 1/10 complete: participant 1, 469 test windows
Fold 2/10 complete: participant 2, 476 test windows
Fold 3/10 complete: participant 3, 468 test windows
Fold 4/10 complete: participant 4, 459 test windows
Fold 5/10 complete: participant 5, 458 test windows
Fold 6/10 complete: participant 6, 435 test windows
Fold 7/10 complete: participant 7, 468 test windows
Fold 8/10 complete: participant 8, 439 test windows
Fold 9/10 complete: participant 9, 461 test windows
Fold 10/10 complete: participant 10, 448 test windows
Out[4]: 2213


## 3.4 Check and summarise the completed experiments
All metrics below are proportions. Pooled metrics combine all held-out predictions; they are not averages of participant scores. Task 4 provides the full comparison, uncertainty and critical discussion.

In [5]:
assert not predictions.duplicated(['model', 'window_id']).any()
assert not predictions.isna().any().any()
prob_columns = [f'p_{label}' for label in CLASSES]
assert np.allclose(predictions[prob_columns].sum(axis=1), 1)
assert predictions[prob_columns].ge(0).all().all()
assert predictions[prob_columns].le(1).all().all()
for name, group in predictions.groupby('model', sort=False):
    aligned = group.set_index('window_id').loc[X.index]
    assert len(aligned) == len(X)
    assert np.array_equal(aligned.true_label, metadata.activity)
    assert np.array_equal(aligned.participant, metadata.participant)
    assert np.array_equal(aligned.fold, metadata.participant)
    assert group.fold.nunique() == 10
    assert np.array_equal(aligned.predicted_label, CLASSES[aligned[prob_columns].to_numpy().argmax(axis=1)])
pooled = pd.DataFrame([{'model': name, **scores(group.true_label, group.predicted_label)}
                       for name, group in predictions.groupby('model', sort=False)])
pooled.to_csv(RESULTS / 'task3_overall_metrics.csv', index=False)
runtime_summary = runtimes.groupby('model', sort=False)[['train_seconds', 'prediction_seconds', 'total_seconds']].sum().reset_index()
runtime_summary = manifest.merge(runtime_summary, on='model', validate='one_to_one')
runtime_summary.to_csv(RESULTS / 'task3_runtime_summary.csv', index=False)
display(pooled.round(4))
display(runtime_summary.round(3))
print('All 70 model/fold evaluations completed. Six forests per fold; decision fusion reuses two.')

All 70 model/fold evaluations completed. Six forests per fold; decision fusion reuses two.


,model,accuracy,macro_precision,macro_recall,macro_f1
0,Chest accelerometer,0.8271,0.8432,0.8355,0.8340
1,Ankle accelerometer,0.8596,0.8781,0.8691,0.8709
2,Wrist accelerometer,0.7396,0.7529,0.7339,0.7374
3,Accelerometers (all locations),0.9681,0.9722,0.9679,0.9691
4,Gyroscopes (ankle + wrist),0.7863,0.7842,0.7486,0.7528
5,Feature fusion,0.9644,0.9685,0.9634,0.9651
6,Decision fusion,0.9644,0.9687,0.9621,0.9645


,model,features,sources,modalities,component_models,train_seconds,prediction_seconds,total_seconds
0,Chest accelerometer,18,1,1,1,19.071,0.108,19.179
1,Ankle accelerometer,18,1,1,1,18.587,0.109,18.697
2,Wrist accelerometer,18,1,1,1,19.438,0.117,19.555
3,Accelerometers (all locations),54,3,1,1,25.403,0.091,25.494
4,Gyroscopes (ankle + wrist),36,2,1,1,23.818,0.107,23.925
5,Feature fusion,90,3,2,1,31.224,0.100,31.325
6,Decision fusion,90,3,2,2,49.220,0.198,49.418


In [6]:
notes = f"""# Task 3: Multimodal fusion modelling

Three physical sources were used: chest, left ankle and right wrist/lower arm. Acceleration and angular velocity were the two selected modalities. Chest acceleration represented trunk movement, while ankle and wrist inputs represented lower-limb and arm movement. These motivated complementary inputs without claiming an observed benefit in advance. ECG and magnetometer inputs were excluded as a scope choice; their inferiority was not tested. Task 2 all-channel window exclusions were preserved.

The source baselines used chest, ankle and wrist acceleration separately (18 features each). The unimodal multi-location baselines used acceleration (54 features) and gyroscopes (36 features). Feature fusion concatenated those inputs into 90 features for a single forest. Decision fusion averaged class-aligned probabilities from the two independently fitted modality forests with fixed weights of 0.5 each. Feature fusion can combine modality features within tree paths; decision fusion combines predictions after separate learning and cannot learn those cross-modal paths.

Every forest used 200 trees, square-root feature subsampling, minimum leaf size 2, balanced training class weights, unlimited depth, seed 42 and one worker. These fixed settings were recorded before evaluation. No tuning or inner validation was performed, and optimality is not claimed. No learned scaling or imputation was required. Model fitting and class weighting used only outer training participants.

All seven comparisons used the same 4,581 windows and ten participant folds from Task 2. Labels, participant identifiers and window identifiers were excluded from predictive matrices. Probability columns were aligned to labels 1–12 and checked to sum to one. Each eligible window has exactly one held-out prediction per experiment. All 70 model/fold evaluations completed with no incomplete experiments.

## Pooled held-out metrics (proportions)
{pooled.to_csv(index=False)}
## Measured computation (seconds, summed over ten folds)
{runtime_summary.to_csv(index=False)}

Training measures fit; prediction measures probability calculation and alignment. Decision-fusion totals include both component costs plus averaging, although components were reused from baseline fits. Feature extraction and file I/O are excluded. Timing is environment-specific; memory, energy and stored model size were not measured. Runtime values are not sums to be added across overlapping experiments.

The offline gap filling, annotation-defined activity boundaries, original all-channel eligibility rule and small participant sample limit interpretation. Task 4 evaluates actual confusion patterns, participant variation and paired cluster uncertainty. The original rubric remains unavailable, so formal HD compliance cannot be certified.
"""
(GENERATED_NOTES / 'task3_multimodal_fusion_modelling.md').write_text(notes)
print('Task 3 results and notes saved. Continue with Task 4.')

Task 3 results and notes saved. Continue with Task 4.
